# Week 07 — Home exercise 6: A function that loads and cleans

**Solution proposal.**

Everything this week has been one job: turn a file on disk into a table you can trust. Here it is as a function, so that next time it is one line.

In [1]:
import numpy as np
import pandas as pd

## The function

In [2]:
def load_emissions(path, drop_missing=True):
    """
    Read the emissions data and return it ready to use.

    Parameters
    ----------
    path : str
        Path to the emissions CSV file.
    drop_missing : bool, optional
        Whether to drop rows with no value for co2_total. Default is True,
        which is what you want for any analysis of emissions. Pass False to
        keep every row, for example when checking how much data is missing.

    Returns
    -------
    DataFrame
        One row per entity per year, with two extra columns: co2_pc, holding
        emissions in tonnes per person, and income_group, holding the World
        Bank income band. income_group is missing where gdp_pc is missing.
    """
    emissions = pd.read_csv(path)

    if drop_missing:
        emissions = emissions.dropna(subset=["co2_total"])

    # co2_total is in millions of tonnes and population is a count of people,
    # so the ratio has to be scaled by a million to come out in tonnes.
    emissions["co2_pc"] = emissions["co2_total"] * 1_000_000 / emissions["population"]

    # The conditions are tried in order and the first match wins, so they have
    # to run from the smallest threshold upwards - the elif ladder again.
    conditions = [
        emissions["gdp_pc"] < 1_000,
        emissions["gdp_pc"] < 10_000,
        emissions["gdp_pc"] < 50_000,
    ]

    emissions["income_group"] = np.select(
        conditions,
        ["low", "lower-middle", "upper-middle"],
        default="high",
    )

    # A missing GDP figure fails every condition and would otherwise be caught
    # by the default and labeled "high". Say what should happen to it instead.
    emissions.loc[emissions["gdp_pc"].isna(), "income_group"] = None

    return emissions

## Checking both settings

In [3]:
emissions = load_emissions("../data/co2_emissions.csv")
everything = load_emissions("../data/co2_emissions.csv", drop_missing=False)

print("with the default:       ", emissions.shape)
print("with drop_missing=False:", everything.shape)

with the default:        (5904, 13)
with drop_missing=False: (6240, 13)


## Did the missing GDP figures fall through?

In [4]:
print("rows with no gdp_pc: ", everything["gdp_pc"].isna().sum())
print("income_group missing:", everything["income_group"].isna().sum())
print()
print(everything["income_group"].value_counts(dropna=False))

rows with no gdp_pc:  166
income_group missing: 166

income_group
lower-middle    2956
upper-middle    1713
low              974
high             431
NaN              166
Name: count, dtype: int64


The two counts match, which is the check worth making.

Without the `.loc` line, those 166 rows would have failed every condition — a comparison with `NaN`
is never true — been caught by `default`, and come out labeled **high income**. No error, no warning,
and 166 of the poorest-documented countries in the dataset filed under the wrong band. Every average
computed by income group after that would have been wrong, and nothing would have said so.

## Highest emissions per person, 2023

In [5]:
year_2023 = emissions[emissions["year"] == 2023]

year_2023.sort_values("co2_pc", ascending=False).head(5)[["country", "code", "co2_pc"]].round(1)

,country,code,co2_pc
4439,Palau,PLW,78.9
4703,Qatar,QAT,48.6
431,Bahrain,BHR,24.5
3047,Kuwait,KWT,22.7
767,Brunei Darussalam,BRN,21.1


## Highest total emissions, 2023

In [6]:
year_2023.sort_values("co2_total", ascending=False).head(5)[["country", "code", "co2_total"]].round(0)

,country,code,co2_total
6167,World,WLD,39113.0
2519,IDA & IBRD total,IBT,26335.0
2495,IBRD only,IBD,25256.0
3407,Low & middle income,LMY,23926.0
3815,Middle income,MIC,23704.0


## Comparing the two lists

**The per-person list is usable.** Palau, Qatar, Bahrain, Kuwait and Brunei are all real countries,
and the result is a genuine finding: emissions per head are dominated by small states with oil, gas or
refining industries and very few people to divide by.

**The total list is not usable at all.** `World`, `IDA & IBRD total`, `IBRD only`,
`Low & middle income` and `Middle income` are World Bank groupings, not countries. They overlap each
other and they contain the countries listed below them, so the ranking is meaningless and adding the
column up would count China several times over.

Why the same data gives one good list and one useless one: dividing by population happens to push the
aggregates down, because a grouping of many countries has an enormous denominator. **That is luck, not
safety.** The aggregates are still in the table, still in every mean, every correlation and every
plot, and the next question you ask might be one where they dominate.

The fix needs a second table. `country_info.csv` marks every entity with a region, and aggregates are
the ones whose region is literally `"Aggregates"`. Bringing the two tables together is the next thing
we learn to do.

### Things worth noticing

- The function **does not print**. It returns a table, and the code that calls it decides what to
  display — the same division of labor as the temperature converter in Part 1, and the reason this
  function can be reused at all.
- `drop_missing` is a parameter rather than a decision baked into the body, because "drop the
  incomplete rows" is right for an analysis and wrong for an audit of how complete the data is. A
  default of `True` makes the common case one word shorter.
- Keeping the function in its own cell, separate from the calls that demonstrate it, is what lets you
  copy it into next week's notebook without dragging five tables along with it.

### What this notebook does NOT do

- It does not remove the aggregates, which is the thing that most needs doing, and it cannot: the
  information is in another file.
- `co2_pc` is computed even when `co2_total` is missing, in the `drop_missing=False` case, and comes
  out as `NaN`. That is the right answer, but it means the column silently has 336 holes in it that
  nothing warns you about.
- It trusts the file's column names completely. If the World Bank renamed a column tomorrow, this
  function would raise a `KeyError` somewhere in the middle rather than saying "the file is not the
  shape I expected".
- It has no tests. `load_emissions("nonsense.csv")` raises a `FileNotFoundError` from pandas rather
  than anything more helpful, and nothing checks that the returned table has the columns it promises.